 # Getting Started: Gemma 4 Developer Agent

## 1. Environment Configuration and Package Installation

In [ ]:
import glob
import importlib
import logging
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['NCCL_DEBUG'] = 'WARN'
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['LITELLM_TELEMETRY'] = 'False'
os.environ['LITELLM_LOG'] = 'WARNING'
os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY', 'EMPTY')
os.environ['LOCAL_API_KEY'] = os.environ.get('LOCAL_API_KEY', 'EMPTY')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['OTEL_PYTHON_DISABLED'] = 'true'


def configure_logging() -> None:
    """Silence verbose third-party loggers."""
    for name in (
        'LiteLLM',
        'LiteLLM Router',
        'LiteLLM Proxy',
        'litellm',
        'httpx',
        'httpcore',
        'openai',
        'urllib3',
        'opentelemetry',
        'opentelemetry.context',
    ):
        lg = logging.getLogger(name)
        lg.setLevel(logging.WARNING)
        if 'litellm' in name.lower():
            lg.propagate = False
            lg.handlers.clear()


configure_logging()

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        try:
            os.symlink(w, target)
        except OSError:
            shutil.copyfile(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
configure_logging()
print('Wheelhouse installation complete.')

## 2. Competition Dataset and Sample Submission

In [ ]:
import re
from swegemma.models import load_tasks

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# Official quantized competition models:
#   - 'gemma-4-31b-it-qat-w4a16-ct' (default in sample_submission)
#   - 'gemma-4-12b-it-qat-w4a16-ct'
#   - 'gemma-4-e4b-it-qat-w4a16-ct'
#   - 'gemma-4-e2b-it-qat-w4a16-ct'
MODEL_NAME = 'gemma-4-12b-it-qat-w4a16-ct'

# ---- Which submission to evaluate / package ----
# Your local `submission/` folder, uploaded as the Kaggle dataset
# "my-agent-submission" and attached to this notebook. If the path below
# doesn't exist, we auto-detect any attached dataset containing an
# agent.yaml (searched recursively, up to 3 levels deep). Set to None to
# force the competition's sample_submission instead.
MY_SUBMISSION_DIR = '/kaggle/input/datasets/dhruvsaraswat907/my-agent-submission'


def _find_submission_root(p: Path, max_depth: int = 3) -> Path:
    """Find the directory containing agent.yaml (up to max_depth levels down)."""
    p = p.resolve()
    if (p / 'agent.yaml').is_file():
        return p
    for depth in range(1, max_depth + 1):
        for match in p.glob('/'.join(['*'] * depth)):
            if match.is_dir() and (match / 'agent.yaml').is_file():
                return match
    raise FileNotFoundError(f'agent.yaml not found under {p} (depth<={max_depth})')


def _auto_detect_submission():
    """Find an attached dataset containing agent.yaml (datasets/<owner>/<slug>/...)."""
    root = Path('/kaggle/input/datasets')
    if not root.is_dir():
        return None
    candidates = []
    for owner in sorted(root.iterdir()):
        if not owner.is_dir():
            continue
        for ds in sorted(owner.iterdir()):
            if not ds.is_dir():
                continue
            try:
                candidates.append(_find_submission_root(ds))
            except FileNotFoundError:
                continue
    candidates.sort(key=lambda p: 0 if 'submission' in p.name else 1)
    return candidates[0] if candidates else None


def _list_tree(p: Path, limit: int = 40) -> None:
    """Print a short file listing so the mounted structure is visible in output."""
    files = [str(f.relative_to(p)) for f in sorted(p.rglob('*')) if f.is_file()]
    for f in files[:limit]:
        print(f'    {f}')
    if len(files) > limit:
        print(f'    ... and {len(files) - limit} more files')


if MY_SUBMISSION_DIR and Path(MY_SUBMISSION_DIR).is_dir():
    SUBMISSION_SRC = _find_submission_root(Path(MY_SUBMISSION_DIR))
    print(f'Using custom submission: {SUBMISSION_SRC}')
    _list_tree(SUBMISSION_SRC)
elif (auto := _auto_detect_submission()) is not None:
    SUBMISSION_SRC = auto
    print(f'WARNING: {MY_SUBMISSION_DIR} not found; auto-detected {SUBMISSION_SRC}')
    _list_tree(SUBMISSION_SRC)
else:
    SUBMISSION_SRC = DATA_DIR / 'sample_submission'
    print(f'ERROR: no custom submission found under /kaggle/input/datasets; '
          f'falling back to sample submission: {SUBMISSION_SRC}')

# Copy the chosen submission to a writable directory for customization and packaging
AGENT_DIR = WORKING_DIR / 'agent_submission'
if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)
shutil.copytree(SUBMISSION_SRC, AGENT_DIR)

# If switching away from 31B, update all agent YAMLs and remove the 31B-shaped sample LoRA adapters
if MODEL_NAME != 'gemma-4-31b-it-qat-w4a16-ct':
    for yaml_path in [AGENT_DIR / 'agent.yaml', *sorted((AGENT_DIR / 'sub_agents').glob('*.yaml'))]:
        if yaml_path.is_file():
            text = yaml_path.read_text(encoding='utf-8')
            text = re.sub(r'^(\s*model:\s*).*$', rf'\g<1>{MODEL_NAME}', text, flags=re.MULTILINE)
            text = re.sub(r'^\s*adapter:\s*.*\n?', '', text, flags=re.MULTILINE)
            yaml_path.write_text(text, encoding='utf-8')
    shutil.rmtree(AGENT_DIR / 'adapters', ignore_errors=True)

# Load tasks from the published dataset
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
tasks = load_tasks(TASKS_PATH)

print(f'Data directory: {DATA_DIR}')
print(f'Agent directory: {AGENT_DIR} (model={MODEL_NAME})')
print(f'Loaded {len(tasks)} tasks from {TASKS_PATH.name}')
for t in tasks[:5]:
    print(f'  - {t.instance_id} ({t.repo} @ {t.base_commit[:8]})')

## 3. Code Graph Functions

In [ ]:
from swegemma import graph as sg

sample_task = tasks[0]
GRAPH_DIR = str(DATA_DIR / 'graphs')
EMBEDDINGS_DIR = str(DATA_DIR / 'embeddings')

# Load the pre-computed repository code graph and node embeddings for the sample task
repo_graph = sg.get_graph(
    repo_name=sample_task.repo,
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    base_commit=sample_task.base_commit,
)
sample_nodes = list(repo_graph.nodes())
print(f'Graph for {sample_task.repo} ({sample_task.instance_id}):')
print(f'  Nodes: {repo_graph.number_of_nodes()}, Edges: {repo_graph.number_of_edges()}')

# Select a connected symbol and query structural neighbors (callers, callees, definitions)
query_node = next((n for n, deg in repo_graph.degree() if deg >= 5), sample_nodes[0])
neighbors = sg.get_neighbor(
    node=query_node,
    graph=repo_graph,
    max_neighbors=10,
)
print(f'\nNeighbors of {query_node!r} (showing up to 5):')
for n in neighbors[:5]:
    print(f'  - {n}')

# Search for semantically similar code nodes using pre-computed vector embeddings
similar = sg.get_similar_nodes(
    node=query_node,
    repo_name=sample_task.repo,
    k=5,
    graph=repo_graph,
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    base_commit=sample_task.base_commit,
)
print(f'\nTop similar nodes to {query_node!r}:')
for item in similar:
    print(f"  - {item['node_name']} (similarity={item['similarity']:.4f})")

# Extract an induced subgraph connecting the query symbol and its neighbors
focal_nodes = [query_node, *neighbors[:4]]
subgraph = sg.get_induced_subgraph(repo_graph, focal_nodes)
print(
    f'\nInduced subgraph over {len(focal_nodes)} focal nodes: '
    f'{subgraph.number_of_nodes()} nodes, {subgraph.number_of_edges()} edges'
)

## 4. Start vLLM Server

In [ ]:
import litellm
import torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from adk_submission.discovery import discover_declared_models
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

litellm.drop_params = True

ALLOWED_MODEL_NAMES = frozenset({
    'gemma-4-31b-it-qat-w4a16-ct',
    'gemma-4-12b-it-qat-w4a16-ct',
    'gemma-4-e4b-it-qat-w4a16-ct',
    'gemma-4-e2b-it-qat-w4a16-ct',
})
MODEL_VLLM_EXTRA_ARGS = {
    'gemma-4-12b-it-qat-w4a16-ct': (
        '--hf-overrides',
        '{"architectures": ["Gemma4ForCausalLM"], "model_type": "gemma4"}',
    ),
}
_SAFE_PATH_COMPONENT = re.compile(r'[A-Za-z0-9][A-Za-z0-9._-]*')
INFERENCE_API_KEY = os.environ.get('LOCAL_API_KEY', 'EMPTY')

# Validate single canonical base model and discover any PEFT LoRA adapters in the submission directory.
# Note: sample_submission includes dummy LoRA adapters (adapters/main_lora and adapters/tool_lora)
# shaped for gemma-4-31b-it-qat-w4a16-ct; Cell 2 removes them automatically when MODEL_NAME != 31B.
declared_models = discover_declared_models(AGENT_DIR)
assert len(declared_models) == 1, f'Expected exactly 1 declared model, got: {sorted(declared_models)}'
declared_model = next(iter(declared_models))
assert declared_model in ALLOWED_MODEL_NAMES, f'Unsupported model {declared_model!r}. Allowed: {sorted(ALLOWED_MODEL_NAMES)}'

MODEL_PATH = Path(f'/kaggle/input/models/google/gemma-4/other/{declared_model}/2')
assert MODEL_PATH.exists(), f'Model weights not found at {MODEL_PATH}. Ensure the model input is attached.'

adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
adapter_map = getattr(adapters, 'adapters', None) or {}
for adapter_name in adapter_map:
    assert _SAFE_PATH_COMPONENT.fullmatch(adapter_name), (
        f'Unsafe adapter name {adapter_name!r}: must match [A-Za-z0-9][A-Za-z0-9._-]*'
    )
detected_ranks = [
    r for info in adapter_map.values() if (r := getattr(info, 'rank', None)) is not None and r > 0
]

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

if 'server_instance' in globals() and server_instance is not None:
    server_instance.stop()

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=bool(adapter_map),
    max_loras=max(1, len(adapter_map)),
    max_lora_rank=max(detected_ranks) if detected_ranks else 128,
    scheduler_reserve_full_isl=False,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
    extra_args=MODEL_VLLM_EXTRA_ARGS.get(declared_model, ()),
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')

# Register model and LoRA adapter aliases for Google ADK
models = server_instance.create_model_registry(
    aliases=[declared_model],
    model_prefix='openai/',
    api_key=INFERENCE_API_KEY,
)

## 5. Run Full Evaluation (All 129 Tasks) with Checkpointing

Runs **every published task** (not just the 2-task sample) with per-task
checkpointing to `/kaggle/working/results_full/task_results.jsonl`, so an
interrupted run (e.g. the 12 h GPU limit) can be resumed by re-running this
cell. Budgets are read from the submission's `eval_config.yaml`.


In [ ]:
import asyncio
import concurrent.futures
import json
import time
import pandas as pd
import yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator


def run_sync(coro_or_fn, *args, **kwargs):
    """Run an async coroutine synchronously inside a notebook event loop."""
    if callable(coro_or_fn) and not asyncio.iscoroutine(coro_or_fn):
        fn = lambda: coro_or_fn(*args, **kwargs)
    else:
        fn = coro_or_fn
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())


# Read optional evaluation settings from the submission's eval_config.yaml / eval_config.yml
eval_config_file = next(
    (c for name in ('eval_config.yaml', 'eval_config.yml') if (c := AGENT_DIR / name).is_file()),
    None,
)
raw_eval_cfg = (
    yaml.safe_load(eval_config_file.read_text(encoding='utf-8')) or {}
    if eval_config_file is not None
    else {}
)
eval_section = raw_eval_cfg.get('evaluation', raw_eval_cfg)

timeout_seconds = int(eval_section.get('timeout_seconds', 300))
verification_timeout_seconds = int(eval_section.get('verification_timeout_seconds', 300))
max_tool_calls = int(eval_section.get('max_tool_calls', 100))
max_time_minutes = float(eval_section.get('max_time_minutes', 60.0))
turns_raw = eval_section.get('max_turns', eval_section.get('max_llm_calls'))
max_turns = int(turns_raw) if turns_raw is not None else None

print(f'Eval budgets: timeout={timeout_seconds}s, tool_calls={max_tool_calls}, '
      f'time={max_time_minutes}min, turns={max_turns}')

# ---- Full evaluation over ALL published tasks (129) with checkpointing ----
ALL_TASKS = tasks
RESULTS_DIR = WORKING_DIR / 'results_full'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT = RESULTS_DIR / 'task_results.jsonl'

# Resume support: skip tasks already completed in a previous (interrupted) run
done = {}
if CHECKPOINT.exists():
    with open(CHECKPOINT) as f:
        for line in f:
            line = line.strip()
            if line:
                rec = json.loads(line)
                done[rec['id']] = rec
    print(f'Resuming: {len(done)} tasks already completed in {CHECKPOINT.name}')

limits, gen_constraints = build_submission_limits()

eval_config = EvalConfig(
    tasks_path=TASKS_PATH,
    snapshots_dir=DATA_DIR / 'snapshots',
    results_dir=RESULTS_DIR,
    submission_dir=AGENT_DIR,
    models=models,
    sandbox='subprocess',
    timeout_seconds=timeout_seconds,
    verification_timeout_seconds=verification_timeout_seconds,
    max_time_minutes=max_time_minutes,
    max_tool_calls=max_tool_calls,
    max_turns=max_turns,
    limits=limits,
    generation_constraints=gen_constraints,
    adapter_manifest=adapters,
    context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=15,
        overlap_size=2,
        token_threshold=14336,
        event_retention_size=5,
    ),
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    wheels_dir=DATA_DIR / 'wheels',
    verbose=False,
)

evaluator = Evaluator(eval_config)
t0 = time.time()
n_total = len(ALL_TASKS)

for idx, task in enumerate(ALL_TASKS, start=1):
    if task.instance_id in done:
        continue
    print(f'[{idx}/{n_total}] Evaluating {task.instance_id} ({task.repo})...', flush=True)
    try:
        result = run_sync(
            evaluator.evaluate_task,
            task=task,
            task_index=idx,
            total_tasks=n_total,
        )
        patch = (result.agent_patch or '').strip()
        rec = {
            'id': task.instance_id,
            'repo': task.repo,
            'resolved': bool(result.resolved),
            'test_exit_code': result.test_exit_code,
            'patch_chars': len(patch),
            'tool_calls': result.tool_calls,
            'duration_seconds': result.duration_seconds,
            'prediction': patch if patch else 'NO_PATCH',
        }
    except Exception as e:  # keep the run alive; record the failure and move on
        rec = {
            'id': task.instance_id,
            'repo': task.repo,
            'resolved': False,
            'test_exit_code': None,
            'patch_chars': 0,
            'tool_calls': None,
            'duration_seconds': None,
            'prediction': 'NO_PATCH',
            'error': f'{type(e).__name__}: {e}',
        }
    # Checkpoint immediately (crash-safe, append-only)
    with open(CHECKPOINT, 'a') as f:
        f.write(json.dumps(rec) + '\n')
    done[task.instance_id] = rec

    n_done = len(done)
    n_resolved = sum(1 for r in done.values() if r['resolved'])
    elapsed = time.time() - t0
    eta = elapsed / n_done * (n_total - n_done) if n_done else 0
    print(
        f'  -> resolved={rec["resolved"]}, patch_chars={rec["patch_chars"]}, '
        f'tool_calls={rec["tool_calls"]}, duration={rec["duration_seconds"]}\n'
        f'  [progress {n_done}/{n_total}] resolved={n_resolved} '
        f'({n_resolved / n_done:.3f}) | elapsed={elapsed / 60:.1f}m | ETA={eta / 60:.1f}m',
        flush=True,
    )

# ---- Final summary ----
rows = [done[t.instance_id] for t in ALL_TASKS if t.instance_id in done]
df = pd.DataFrame(rows)
resolved = int(df['resolved'].sum())
print(f'\n=== FULL EVALUATION SUMMARY ===')
print(f'Tasks evaluated : {len(df)}/{n_total}')
print(f'Resolved        : {resolved}')
print(f'Resolution rate : {resolved / len(df):.4f}')
print(f'Total wall time : {(time.time() - t0) / 60:.1f} min')

# Per-repo breakdown
per_repo = df.groupby('repo')['resolved'].agg(['sum', 'count', 'mean']).sort_values('mean', ascending=False)
print('\nPer-repo resolution rate:')
print(per_repo.to_string())

# Failure-mode breakdown (for the paper's failure analysis)
no_patch = int((df['prediction'] == 'NO_PATCH').sum())
bad_patch = int(((df['prediction'] != 'NO_PATCH') & (~df['resolved'])).sum())
print(f'\nFailure modes: no_patch={no_patch}, patch_but_failed={bad_patch}, resolved={resolved}')

# Submission-format parquet (id, prediction) — same shape the scorer expects
submission_df = df[['id', 'prediction']].copy()
submission_df.to_parquet(RESULTS_DIR / 'submission.parquet', index=False)
df.to_json(RESULTS_DIR / 'summary.jsonl', orient='records', lines=True)
print(f'\nSaved: {RESULTS_DIR / "submission.parquet"}')
print(f'Saved: {RESULTS_DIR / "summary.jsonl"}')
display(submission_df.head(10))


## 6. Package Submission Archive

In [ ]:
import zipfile
from adk_submission import validate_directory
from swegemma.config import ALLOWED_SUBMISSION_EXTENSIONS, MAX_SUBMISSION_SIZE_BYTES

# Validate unpacked submission directory structure, file types, symlinks, and size limits
for dirpath, dirnames, filenames in os.walk(AGENT_DIR, followlinks=False):
    parent = Path(dirpath)
    for name in (*dirnames, *filenames):
        assert not (parent / name).is_symlink(), f'Symlinks are not permitted: {(parent / name).relative_to(AGENT_DIR)}'
validate_directory(AGENT_DIR, limits=limits)

# Create submission.zip with agent.yaml at the root of the archive
zip_base = WORKING_DIR / 'submission'
zip_path = Path(shutil.make_archive(str(zip_base), 'zip', root_dir=AGENT_DIR))

# Validate archive contents against competition constraints
with zipfile.ZipFile(zip_path, 'r') as zf:
    infos = zf.infolist()
    total_size = sum(i.file_size for i in infos)
    assert total_size <= MAX_SUBMISSION_SIZE_BYTES, f'Archive exceeds 3 GiB limit: {total_size}'
    for info in infos:
        if not info.is_dir():
            ext = Path(info.filename).suffix.lower()
            assert ext in ALLOWED_SUBMISSION_EXTENSIONS, f'Disallowed file extension: {info.filename}'

print(f'Created {zip_path} ({zip_path.stat().st_size / (1024 * 1024):.2f} MiB)')